In [1]:
import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]


# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")
     

Client ready.


In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What is doppler effect?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage)

The Doppler effect is a fundamental concept in physics that describes the change in frequency or pitch of a wave that occurs when the source of the wave and the observer are moving relative to each other. This effect is named after the Austrian physicist Christian Doppler, who first described it in 1842.

The Doppler effect can be observed in various types of waves, including sound waves, light waves, and other forms of electromagnetic radiation. Here's how it works:

**When the source and observer are moving towards each other:**

* The frequency of the wave increases (or the pitch becomes higher)
* The wavelength of the wave decreases

**When the source and observer are moving away from each other:**

* The frequency of the wave decreases (or the pitch becomes lower)
* The wavelength of the wave increases

The Doppler effect has many practical applications in various fields, including:

1. **Radar technology**: The Doppler effect is used to measure the speed of objects, such as cars 

## Student Reasoning — Anatomy of a Call

**1. Difference between system and user**

- **system**: Tells the AI how to behave or what role to take.
  - Example: `"You are a helpful math tutor."`
- **user**: Contains the actual question or instruction we want the AI to respond to.
  - Example: `"Simple explain what eigenvalues and eigenvectors are."`

**2. What is a token?**

A **token** is a small piece of text that an LLM processes, such as a word, or part of a word.

**Why do API providers bill per token rather than per request?**

API providers bill per token because **different requests use different amounts of text**. Tokens provide a better measure of how much the model processes and generates.

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0
# and 5 times at temperature=1.2.

# TODO: Print all 10 answers, grouped by temperature.
question = "What is core banking system?"

print("--- Temperature 0.0 ---")
for i in range(5):
  answer = ask_llm(question, temperature=0.0)
  print(f"{i + 1}. {answer.choices[0].message.content}")

print("\n--- Temperature 1.2 ---")
for i in range(5):
  answer = ask_llm(question, temperature=1.2)
  print(f"{i + 1}. {answer.choices[0].message.content}")

--- Temperature 0.0 ---
1. A core banking system (CBS) is a software application that enables banks and other financial institutions to manage their daily operations, customer accounts, and financial transactions efficiently. It is the backbone of a bank's information technology (IT) infrastructure and provides a centralized platform for processing transactions, managing accounts, and delivering banking services to customers.

A typical core banking system includes the following components:

1. **Account management**: CBS manages customer accounts, including account opening, account maintenance, and account closure.
2. **Transaction processing**: CBS processes various types of transactions, such as deposits, withdrawals, transfers, and payments.
3. **Ledger management**: CBS maintains a centralized ledger that records all transactions, allowing for real-time updates and reconciliation.
4. **Loan management**: CBS manages loan accounts, including loan origination, loan servicing, and lo

## Student Reasoning — Temperature

- **Temperature = 0.0:** The answers were very similar. The model consistently explained core banking systems using almost the same structure and ideas.


- **Temperature = 1.2:** The answers were more varied. The model used different wording, examples, and structures, and one response even reached the `max_tokens` limit.

For the **loan decision-support system**, a **low temperature (around 0.0–0.3)** is more appropriate because decisions should be consistent, predictable, and less influenced by randomness.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.
